# Gradio Day!

Company Brochure, with the option to select the model

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
#from scraper import fetch_website_contents
import gradio as gr

In [ ]:
from scraper import fetch_website_contents

In [ ]:
# Load environment variables

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

In [ ]:
# Connect to OpenAI, Ollama and Google

openai = OpenAI()
#openai_client = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

gemini_client = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
gpt_model = "gpt-4.1-mini"
gemini_model = "gemini-2.5-flash-lite"
ollama_model = "llama3.2:latest"

In [ ]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
def stream_ollama(prompt):
    messages = [{"role":"system","content":system_message},{"role":"user","content":prompt}]
    strm_resp = ollama_client.chat.completions.create(model = ollama_model,messages=messages, stream=True)

    result = ""
    for chunk in strm_resp:
        result += chunk.choices[0].delta.content or ""
    yield result
    

In [ ]:
def stream_gem(prompt):
    messages = [{"role":"system","content":system_message},{"role":"user","content":prompt}]
    strm_resp = gemini_client.chat.completions.create(model = gemini_model,messages=messages, stream=True)

    result = ""
    for chunk in strm_resp:
        result += chunk.choices[0].delta.content or ""
    yield result

In [ ]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="Ollama":
        result = stream_ollama(prompt)
    elif model=="Gemini":
        result = stream_gem(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result
    

In [ ]:
name_input = gr.Textbox(label = "Company Name")
url_input = gr.Textbox(label = "Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["Ollama","Gemini"],label = "Select Model",value = "Ollama")
message_output = gr.Markdown(label = "Response:")

view = gr.Interface(
    fn= stream_brochure,
    title = "Brochure Generator",
    inputs=[name_input,url_input,model_selector],
    outputs= [message_output],
    examples = [
        ["OrbitShift", "https://orbitshift.com", "Ollama"],
        ["Edward Donner", "https://edwarddonner.com", "Gemini"]
    ],
    flagging_mode="never"
)
view.launch(inbrowser=True)